# 7장. RAG 시스템 평가 — 03. LangSmith 데이터셋과 LLM-as-Judge 평가

## 학습 목표

- **LangSmith(랭스미스)**에 평가 데이터셋을 업로드하고 관리하는 방법 이해
- **LLM-as-Judge** 패턴으로 자동 평가 구현
- 임베딩(Embedding) 유사도 기반 평가 방법 적용

## 사전 지식

- LangSmith 계정 및 API 키 설정 (`.env`에 `LANGCHAIN_API_KEY`)
- RAGAS 평가 경험 (02번 노트북)

## LangSmith 평가 워크플로우

```mermaid
flowchart LR
    A[📊 평가 데이터셋<br/>inputs + outputs] -->|create_dataset| B[LangSmith<br/>클라우드 저장]
    B --> C[evaluate 함수<br/>실행]
    D[🤖 RAG 시스템<br/>평가 대상] --> C
    E[⚖️ 평가자<br/>Evaluator] --> C
    C --> F[실험 결과<br/>Experiment]
    F --> G[📈 LangSmith<br/>대시보드]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A input
    class D,E process
    class B,F storage
    class C process
    class G output
```

---

## 환경 설정

In [1]:
# 필요한 패키지 설치
# !pip install -qU langsmith langchain langchain-openai


In [2]:
from dotenv import load_dotenv
import os

# 환경변수 로드
load_dotenv()

# macOS에서 FAISS 사용 시 OpenMP 중복 로드 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# LangSmith 프로젝트 설정
os.environ["LANGSMITH_PROJECT"] = "03-Eval-Dataset-Evaluation"

# LangSmith 설정 확인
print(f"LANGCHAIN_API_KEY: {'설정됨' if os.getenv('LANGCHAIN_API_KEY') else '미설정'}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT')}")

# ---------------------------------------------------
# [LangSmith UI 확인 방법]
# 1. https://smith.langchain.com 접속
# 2. 좌측 Projects 클릭
# 3. "Eval-Dataset-Evaluation" 프로젝트 선택
# 4. 주안점: Datasets 메뉴 → 데이터셋 생성 확인, Experiments 탭 → 평가 점수 확인
# ---------------------------------------------------

LANGCHAIN_API_KEY: 미설정
LANGSMITH_PROJECT: 03-Eval-Dataset-Evaluation


---

## LangSmith 데이터셋 생성

LangSmith에서 데이터셋은 `inputs`와 `outputs`(선택)로 구성돼요.

- **inputs**: 시스템에 전달할 입력 (예: `{"question": "..."}`)
- **outputs**: 기대하는 정답 (예: `{"answer": "..."}`) — 평가자가 비교 기준으로 사용

> **실무 팁**: 동일한 데이터셋을 여러 실험(experiment)에서 재사용할 수 있어요. 모델이나 청크 크기를 바꾸면서 같은 데이터셋으로 비교하는 것이 좋은 관행입니다.

> 🎯 **강의 포인트**: LangSmith 데이터셋은 `inputs`와 `outputs`으로 구성됩니다. 핵심은 **재사용성**입니다. 모델을 바꾸거나 파라미터를 조정할 때마다 같은 데이터셋을 사용해야 공정한 비교가 가능합니다. 데이터셋을 클라우드에 저장하는 순간, 그것이 평가의 기준이 됩니다.

In [ ]:
# ---------------------------------------------------
# 평가용 질문-답변 쌍 준비 및 데이터프레임 생성
# ---------------------------------------------------
import pandas as pd
from langsmith import Client

# LangSmith 클라이언트 초기화 — LANGCHAIN_API_KEY 환경변수를 자동으로 읽음
client = Client()

# 평가용 질문과 답변 준비
# 일부러 틀린 참조 답변(❌)을 섞어서, 평가자가 이를 구분하는지 확인합니다
inputs = [
    "트랜스포머 아키텍처란 무엇인가요?",
    "셀프 어텐션 메커니즘은 어떻게 작동하나요?",
    "트랜스포머가 RNN보다 유리한 점은 무엇인가요?",
    "트랜스포머의 인코더-디코더 구조를 설명해주세요.",
    "멀티헤드 어텐션이란 무엇인가요?",
    "포지셔널 인코딩의 역할은 무엇인가요?",
    "스케일드 닷 프로덕트 어텐션이란 무엇인가요?",
]

outputs = [
    # ✅ 올바른 답변
    "트랜스포머는 순환이나 합성곱 없이 어텐션 메커니즘에만 전적으로 의존하는 신경망 아키텍처입니다.",
    # ✅ 올바른 답변
    "셀프 어텐션은 시퀀스 내 모든 위치 간의 어텐션 점수를 계산하여, 모델이 각 부분의 중요도를 가중치로 반영할 수 있게 합니다.",
    # ❌ 틀린 답변: 실제로는 병렬 처리가 장점인데 순차 처리가 장점이라고 함
    "트랜스포머는 RNN처럼 시퀀스를 순차적으로 처리하며, 단거리 의존성에 특화되어 있어 RNN보다 느리지만 정확합니다.",
    # ✅ 올바른 답변
    "트랜스포머는 입력을 처리하는 인코더와 출력을 생성하는 디코더로 구성되며, 둘 다 멀티헤드 어텐션을 사용합니다.",
    # ❌ 틀린 답변: 멀티헤드가 아니라 단일 어텐션이라고 설명
    "멀티헤드 어텐션은 하나의 어텐션 함수만 사용하여, 단일 표현 공간에서 정보를 순차적으로 처리하는 기법입니다.",
    # ✅ 올바른 답변
    "포지셔널 인코딩은 트랜스포머에 단어의 위치 정보를 제공하여, 순환 구조 없이도 시퀀스의 순서를 파악할 수 있게 합니다.",
    # ✅ 올바른 답변
    "스케일드 닷 프로덕트 어텐션은 쿼리와 키의 내적을 키 차원의 제곱근으로 나눈 뒤 소프트맥스를 적용하여 어텐션 가중치를 계산하는 방식입니다.",
]

# 데이터프레임 생성
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

print(f"평가 데이터: {len(df)}개 (올바른 답변 5개, 틀린 답변 2개)")
df

### LangSmith에 데이터셋 업로드

`read_dataset`으로 기존 데이터셋을 확인하고, 없으면 `create_dataset`으로 새로 만들어요. 동일한 이름의 데이터셋이 중복 생성되지 않도록 예외 처리를 합니다.

> ⚠️ **자주 하는 실수**: `create_dataset()`을 반복 실행하면 동일 이름의 데이터셋이 중복 생성됩니다. `try/except LangSmithNotFoundError` 패턴으로 기존 데이터셋을 먼저 확인하고, 없을 때만 생성하는 습관을 들이세요. 실습 환경에서 셀을 여러 번 실행할 때 특히 주의하세요.

In [4]:
# ---------------------------------------------------
# LangSmith에 데이터셋 업로드 (중복 생성 방지 처리 포함)
# ---------------------------------------------------
# ============================================================
# TODO: 데이터셋이 없으면 create_dataset()과 create_example()로 생성하세요
# 힌트: try: client.read_dataset(...) except LangSmithNotFoundError: client.create_dataset(...)
# 예상 결과: "기존 데이터셋 사용" 또는 "새 데이터셋 생성" 출력
# ============================================================

from langsmith import utils as ls_utils

# 데이터셋 이름
dataset_name = "transformer-qa-evaluation"

# 데이터셋 생성 또는 기존 데이터셋 사용
try:
    # 기존 데이터셋 확인
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"기존 데이터셋 사용: {dataset_name}")
# 구체적 예외 타입을 지정하면 예상치 못한 오류(네트워크 장애 등)를 놓치지 않습니다
except ls_utils.LangSmithNotFoundError:
    # 새 데이터셋 생성
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="트랜스포머 아키텍처 Q&A 평가 데이터셋"
    )
    print(f"새 데이터셋 생성: {dataset_name}")
    
    # 데이터 추가 — inputs/outputs 형식으로 각 예제 업로드
    for q, a in zip(inputs, outputs):
        client.create_example(
            inputs={"question": q},
            outputs={"answer": a},
            dataset_id=dataset.id
        )
    print(f"✅ {len(inputs)}개의 예제 추가 완료")

기존 데이터셋 사용: transformer-qa-evaluation


---

## 평가할 RAG 시스템 구축

In [5]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ---------------------------------------------------
# 트랜스포머 관련 한국어 컨텍스트 문서
# ---------------------------------------------------
documents = [
    Document(page_content=(
        "트랜스포머(Transformer)는 2017년 구글이 'Attention Is All You Need' 논문에서 "
        "제안한 신경망 아키텍처입니다. 기존의 순환 신경망(RNN)이나 합성곱 신경망(CNN) 없이 "
        "어텐션 메커니즘만으로 시퀀스를 처리합니다. 인코더와 디코더 각각 6개의 동일한 레이어를 "
        "쌓아 올린 구조이며, 모든 레이어의 출력 차원은 d_model = 512입니다."
    )),
    Document(page_content=(
        "셀프 어텐션(Self-Attention)은 시퀀스 내 모든 위치 간의 관계를 계산하는 메커니즘입니다. "
        "입력에서 쿼리(Query), 키(Key), 값(Value) 벡터를 생성하고, 쿼리와 키의 유사도로 "
        "어텐션 가중치를 계산합니다. 이를 통해 모델이 입력의 어떤 부분에 집중해야 하는지 학습합니다."
    )),
    Document(page_content=(
        "멀티헤드 어텐션(Multi-Head Attention)은 셀프 어텐션을 여러 개의 '헤드'로 병렬 수행하는 "
        "기법입니다. 각 헤드는 서로 다른 표현 부분공간(representation subspace)에서 정보를 추출합니다. "
        "논문에서는 8개의 헤드를 사용했으며, 각 헤드의 출력을 연결한 뒤 선형 변환을 적용합니다."
    )),
    Document(page_content=(
        "포지셔널 인코딩(Positional Encoding)은 트랜스포머에 단어의 위치 정보를 제공합니다. "
        "트랜스포머는 순환 구조가 없어 단어의 순서를 알 수 없으므로, 사인(sin)과 코사인(cos) "
        "함수 기반의 위치 벡터를 입력 임베딩에 더해줍니다."
    )),
    Document(page_content=(
        "트랜스포머의 인코더는 입력 시퀀스를 연속적인 표현으로 변환하고, 디코더는 이를 바탕으로 "
        "출력을 한 토큰씩 생성합니다. 디코더에는 마스크드 셀프 어텐션이 추가되어 미래 토큰 참조를 "
        "방지합니다. 인코더-디코더 어텐션으로 디코더가 입력의 관련 부분에 집중합니다."
    )),
    Document(page_content=(
        "트랜스포머는 RNN과 달리 시퀀스를 병렬로 처리할 수 있어 훈련 속도가 크게 빠릅니다. "
        "RNN은 순차 처리가 필수지만 트랜스포머는 모든 위치를 동시에 처리합니다. "
        "장거리 의존성도 더 효과적으로 포착할 수 있습니다."
    )),
    Document(page_content=(
        "스케일드 닷 프로덕트 어텐션은 쿼리와 키의 내적을 키 차원의 제곱근으로 나누어 스케일링합니다. "
        "스케일링 없이는 내적 값이 커져 소프트맥스의 기울기가 매우 작아집니다. "
        "스케일링 후 소프트맥스를 적용하여 가중치를 구하고, 값 벡터에 곱하여 출력을 얻습니다."
    )),
]

# 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

# 3단계: 프롬프트 정의
prompt = PromptTemplate.from_template(
    """주어진 컨텍스트를 바탕으로 질문에 답변하세요. 간결하고 구체적으로 답변하세요.

컨텍스트: {context}

질문: {question}

답변:"""
)

# 4단계: LLM 및 체인 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG 시스템 구축 완료")

RAG 시스템 구축 완료


### RAG 시스템 동작 확인

> 🔑 **핵심 개념**: LLM-as-Judge는 강력하지만 편향이 있습니다. 생성 LLM과 평가 LLM이 같은 모델이면 자기 자신의 답변에 후한 점수를 주는 경향이 있습니다. 가능하면 생성은 소형 모델(gpt-4o-mini)로, 평가는 더 강력한 모델로 분리하는 것이 좋은 관행입니다.

In [6]:
# 테스트 질문
test_q = "트랜스포머 아키텍처란 무엇인가요?"
answer = rag_chain.invoke(test_q)

print("질문:", test_q)
print("\n답변:")
print(answer)

질문: 트랜스포머 아키텍처란 무엇인가요?

답변:
트랜스포머 아키텍처는 2017년 구글이 제안한 신경망 구조로, 어텐션 메커니즘만으로 시퀀스를 처리합니다. 인코더와 디코더 각각 6개의 동일한 레이어로 구성되며, 모든 레이어의 출력 차원은 512입니다. RNN이나 CNN 없이 병렬 처리가 가능하여 훈련 속도가 빠르고, 장거리 의존성을 효과적으로 포착할 수 있습니다.


---

## LLM-as-Judge 평가

**LLM-as-Judge**는 LLM을 평가자로 활용해 답변 품질을 자동으로 판단하는 방식이에요. 사람이 평가하는 것과 유사한 정성적 판단을 자동화할 수 있어요.

LangSmith는 다양한 사전 정의된 평가자를 제공해요:

| 평가자 | 설명 | 참조 답변 필요 |
|--------|------|:--------------:|
| `qa` | 질문-답변 쌍의 정확성 평가 (가장 기본) | O |
| `cot_qa` | Chain-of-Thought 추론 후 정확성 평가 | O |
| `context_qa` | 검색된 Context 기반으로 정확성 평가 | O (Context) |
| `labeled_criteria` | 참조 답변과 비교하여 기준 충족 여부 평가 | O |
| `criteria` | 참조 없이 특정 기준으로 평가 | X |
| `embedding_distance` | 임베딩 벡터 거리로 유사도 평가 | O |

> **실무 팁**: LLM-as-Judge는 편향이 생길 수 있어요. 평가 LLM과 생성 LLM이 같은 경우 자기 자신의 답변에 높은 점수를 주는 경향이 있습니다. 가능하면 더 강력한 모델(GPT-4 계열)을 평가자로 사용하세요.

In [7]:
# ---------------------------------------------------
# LLM-as-Judge 평가자 설정
# ---------------------------------------------------
# ============================================================
# TODO: LangChainStringEvaluator("qa", ...)와 LangChainStringEvaluator("labeled_criteria", ...)로 평가자를 만드세요
# 힌트: prepare_data=lambda run, example: {"prediction": run.outputs["output"], "reference": example.outputs.get("answer", ""), ...}
# 예상 결과: "QA 평가자 + Criteria 평가자 3개 설정 완료" 출력
# ============================================================

from langsmith.evaluation import LangChainStringEvaluator, evaluate

# 평가용 함수 정의
def rag_qa_function(inputs: dict) -> dict:
    """RAG 시스템에 질문하고 답변 반환"""
    question = inputs["question"]
    answer = rag_chain.invoke(question)
    return {"output": answer}

# --- 1) QA 평가자: 가장 기본적인 질문-답변 정확성 평가 ---
# "qa" 평가자는 질문, 생성 답변, 참조 답변을 비교하여 CORRECT/INCORRECT 판정
#
# prepare_data 파라미터:
#   LangSmith 내부 데이터를 평가자가 요구하는 형식으로 변환하는 함수
#   - run: 평가 대상 시스템(rag_qa_function)의 실행 결과 객체
#          run.outputs → rag_qa_function이 반환한 딕셔너리 (예: {"output": "답변 텍스트"})
#   - example: 데이터셋에 등록된 하나의 예제 객체
#          example.inputs  → 데이터셋의 입력 (예: {"question": "..."})
#          example.outputs → 데이터셋의 참조 답변 (예: {"answer": "..."})
#
# 반환 딕셔너리의 키:
#   - "prediction": 시스템이 생성한 답변 (평가 대상)
#   - "reference": 데이터셋의 참조 답변 (정답 기준)
#   - "input": 원래 질문 (평가자가 맥락 파악에 사용)
qa_evaluator = LangChainStringEvaluator(
    "qa",
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs.get("answer", ""),
        "input": example.inputs["question"],
    }
)

# --- 2) Labeled Criteria 평가자: 기준별로 각각 생성 ---
# labeled_criteria는 한 번에 하나의 기준(criterion)만 평가할 수 있어요.
# 여러 기준을 평가하려면 기준별로 별도의 평가자 인스턴스를 만들어야 합니다.
#
# config 파라미터:
#   - "criteria": 평가 기준을 {이름: 설명} 딕셔너리로 전달
#     평가 LLM이 이 설명을 보고 해당 기준에 맞는지 판단합니다
criteria_definitions = {
    "accuracy": "참조 답변과 비교하여 사실적으로 정확한 답변인가?",
    "relevance": "질문에 직접적으로 답변하고 있는가?",
    "completeness": "답변이 충분히 포괄적인가?",
}

criteria_evaluators = []
for criterion_name, criterion_desc in criteria_definitions.items():
    evaluator = LangChainStringEvaluator(
        "labeled_criteria",
        config={
            "criteria": {criterion_name: criterion_desc},
        },
        prepare_data=lambda run, example: {
            "prediction": run.outputs["output"],
            "reference": example.outputs.get("answer", ""),
            "input": example.inputs["question"],
        }
    )
    criteria_evaluators.append(evaluator)

# 전체 평가자 리스트: QA + Criteria 3개 = 총 4개
all_evaluators = [qa_evaluator] + criteria_evaluators

print(f"QA 평가자 + Criteria 평가자 {len(criteria_evaluators)}개 설정 완료 (총 {len(all_evaluators)}개)")

QA 평가자 + Criteria 평가자 3개 설정 완료 (총 4개)


> ⚠️ **자주 하는 실수**: `labeled_criteria`에 여러 기준을 딕셔너리로 한꺼번에 전달하면(`{"accuracy": "...", "relevance": "...", "completeness": "..."}`) **마지막 기준 하나만 평가**됩니다. 내부적으로 단일 criterion만 처리하도록 설계되어 있기 때문이에요. 반드시 **기준별로 별도의 평가자 인스턴스**를 만들어야 합니다.

> 🔑 **핵심 개념**: `qa` 평가자는 가장 기본적인 LLM-as-Judge 평가자입니다. 질문(input), 생성 답변(prediction), 참조 답변(reference)을 비교해서 CORRECT/INCORRECT를 판정해요. 비슷한 계열로 `cot_qa`(Chain-of-Thought 추론 후 판정)와 `context_qa`(검색된 문서 기반 판정)가 있습니다.

> 💡 **실무 팁**: 임베딩 거리 평가는 LLM 호출 없이 의미적 유사도를 측정하므로 **비용이 거의 들지 않습니다**. 빠른 반복 개발 중에는 임베딩 평가자로 먼저 스크리닝하고, 유망한 설정에만 LLM-as-Judge를 적용하는 2단계 전략이 효율적입니다.

### 평가 실행

In [8]:
# ---------------------------------------------------
# LangSmith evaluate() 실행 — 결과는 대시보드에서 확인 가능
# ---------------------------------------------------
# 평가 실행: QA 평가자 1개 + Criteria 평가자 3개 = 총 4개
experiment_results = evaluate(
    rag_qa_function,
    data=dataset_name,
    evaluators=all_evaluators,
    experiment_prefix="rag-transformer-qa",
    # metadata: LangSmith 대시보드에서 실험 조건을 구별하는 데 사용
    metadata={
        "model": "gpt-4o-mini",
        "chunk_size": 1000,
        "embedding": "text-embedding-3-small"
    }
)

print("\n평가 완료!")
print(f"실험 이름: {experiment_results.experiment_name}")

View the evaluation results for experiment: 'rag-transformer-qa-0f853f2c' at:
https://smith.langchain.com/o/7f9bfcb6-cd10-4707-bfdd-8aeff8fa7bb6/datasets/6458186a-4a71-4460-985f-a559f138340f/compare?selectedSessions=c9e570c9-0a40-4e49-98ba-70da33f3ebea




0it [00:00, ?it/s]


평가 완료!
실험 이름: rag-transformer-qa-0f853f2c


---

## 임베딩 유사도 평가

**임베딩 거리** 기반 평가는 참조 답변과 생성 답변을 벡터로 변환한 뒤 코사인 거리를 측정해요. LLM 호출 없이 의미적 유사도를 빠르게 측정할 수 있습니다.

In [9]:
# ---------------------------------------------------
# 임베딩 거리 기반 평가자 설정 및 실행
# ---------------------------------------------------
# ============================================================
# TODO: "embedding_distance" 평가자를 만들고 evaluate()를 실행하세요
# 힌트: LangChainStringEvaluator("embedding_distance", config={"embeddings": embeddings, "distance_metric": EmbeddingDistance.COSINE})
# 예상 결과: embedding_results.experiment_name 출력
# ============================================================

from langchain.evaluation import EmbeddingDistance
from langsmith.evaluation import evaluate

# 임베딩 거리 평가자 — 코사인 거리로 의미적 유사도 측정 (LLM 호출 없음)
embedding_evaluator = LangChainStringEvaluator(
    "embedding_distance",
    config={
        "embeddings": embeddings,
        "distance_metric": EmbeddingDistance.COSINE
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs.get("answer", ""),
    }
)

# 평가 실행
embedding_results = evaluate(
    rag_qa_function,
    data=dataset_name,
    evaluators=[embedding_evaluator],
    experiment_prefix="rag-embedding-similarity"
)

print("임베딩 유사도 평가 완료!")
print(f"실험 이름: {embedding_results.experiment_name}")

View the evaluation results for experiment: 'rag-embedding-similarity-552a8f60' at:
https://smith.langchain.com/o/7f9bfcb6-cd10-4707-bfdd-8aeff8fa7bb6/datasets/6458186a-4a71-4460-985f-a559f138340f/compare?selectedSessions=20d823ae-dde1-4d62-95ae-4d6bcbbc3d4b




0it [00:00, ?it/s]

임베딩 유사도 평가 완료!
실험 이름: rag-embedding-similarity-552a8f60


---

## 평가 결과 해석

LangSmith 대시보드에서 결과를 확인할 수 있지만, 코드에서도 직접 결과를 가져와 분석할 수 있어요. 각 평가 지표가 무엇을 의미하는지 실제 데이터로 확인해볼게요.

> 🎯 **강의 포인트**: 이 데이터셋에는 **일부러 틀린 참조 답변 2개**를 넣었습니다. RAG 시스템은 컨텍스트에 근거하여 올바르게 답변하지만, 참조 답변이 틀리기 때문에 평가자가 이를 어떻게 판정하는지 관찰하세요.

In [ ]:
# ---------------------------------------------------
# 평가 결과 상세 분석 — QA + Criteria 평가
# ---------------------------------------------------
import pandas as pd

# experiment_results에서 개별 결과 추출
rows = []
for result in experiment_results:
    question = result["example"].inputs["question"]
    reference = result["example"].outputs.get("answer", "")[:40] + "..."
    prediction = result["run"].outputs.get("output", "")[:40] + "..."
    
    # 각 평가자의 결과 수집
    scores = {}
    for eval_result in result["evaluation_results"]["results"]:
        key = eval_result.key
        if eval_result.score is not None:
            scores[key] = eval_result.score
        elif eval_result.value:
            scores[key] = eval_result.value
    
    rows.append({
        "질문": question[:25] + "...",
        "correctness": scores.get("correctness", "-"),
        "accuracy": scores.get("accuracy", "-"),
        "completeness": scores.get("completeness", "-"),
        "relevance": scores.get("relevance", "-"),
    })

result_df = pd.DataFrame(rows)
print("=" * 80)
print("QA + Criteria 평가 결과 (질문별)")
print("=" * 80)
print(result_df.to_string(index=False))

# 요약 통계
print("\n" + "-" * 80)
print("요약:")
correct_count = sum(1 for r in rows if r["correctness"] == "CORRECT")
incorrect_count = sum(1 for r in rows if r["correctness"] == "INCORRECT")
print(f"  correctness:  CORRECT {correct_count} / INCORRECT {incorrect_count}")
acc_y = sum(1 for r in rows if r["accuracy"] == "Y")
acc_n = sum(1 for r in rows if r["accuracy"] == "N")
print(f"  accuracy:     Y {acc_y} / N {acc_n}")
comp_y = sum(1 for r in rows if r["completeness"] == "Y")
comp_n = sum(1 for r in rows if r["completeness"] == "N")
print(f"  completeness: Y {comp_y} / N {comp_n}")
rel_y = sum(1 for r in rows if r["relevance"] == "Y")
print(f"  relevance:    Y {rel_y} / N {len(rows) - rel_y}")


In [ ]:
# ---------------------------------------------------
# 평가 결과 상세 분석 — 임베딩 유사도 평가
# ---------------------------------------------------
emb_rows = []
for result in embedding_results:
    question = result["example"].inputs["question"]
    
    for eval_result in result["evaluation_results"]["results"]:
        if eval_result.key == "embedding_cosine_distance":
            distance = eval_result.score
    
    emb_rows.append({
        "질문": question[:30] + "...",
        "cosine_distance": f"{distance:.4f}",
        "유사도 (1-distance)": f"{1 - distance:.4f}",
    })

emb_df = pd.DataFrame(emb_rows)
print("=" * 70)
print("임베딩 코사인 거리 (질문별)")
print("=" * 70)
print(emb_df.to_string(index=False))

avg_dist = sum(float(r["cosine_distance"]) for r in emb_rows) / len(emb_rows)
print(f"\n평균 코사인 거리: {avg_dist:.4f} (유사도: {1 - avg_dist:.4f})")
print("\n→ 거리가 0에 가까울수록 생성 답변과 참조 답변의 의미가 유사합니다")
print("→ 틀린 참조 답변이 포함된 질문은 거리가 더 크게 나올 수 있습니다")


### 실제 실행 결과 해석

위 코드를 실행하면 다음과 유사한 결과를 얻을 수 있습니다 (실행 환경에 따라 약간 다를 수 있어요):

**QA + Criteria 평가 결과:**

| 질문 | correctness | accuracy | completeness | relevance |
|------|:-----------:|:--------:|:------------:|:---------:|
| 트랜스포머 아키텍처란? | CORRECT | Y | Y | Y |
| 셀프 어텐션 메커니즘은? | CORRECT | Y | Y | Y |
| **트랜스포머가 RNN보다 유리한 점은?** | **INCORRECT** | **N** | **N** | Y |
| 인코더-디코더 구조를 설명해주세요 | CORRECT | Y | Y | Y |
| **멀티헤드 어텐션이란?** | **INCORRECT** | **N** | Y | Y |
| 포지셔널 인코딩의 역할은? | CORRECT | Y | N | Y |
| 스케일드 닷 프로덕트 어텐션이란? | CORRECT | Y | Y | Y |

**임베딩 코사인 거리:**

| 질문 | cosine_distance | similarity |
|------|:-----------:|:----------:|
| 스케일드 닷 프로덕트 어텐션이란? | 0.0966 | 0.9034 |
| 트랜스포머가 RNN보다 유리한 점은? (❌ 틀린 참조) | 0.1869 | 0.8131 |
| 포지셔널 인코딩의 역할은? | 0.2920 | 0.7080 |
| 멀티헤드 어텐션이란? (❌ 틀린 참조) | 0.2901 | 0.7099 |
| 트랜스포머 아키텍처란? | 0.4786 | 0.5214 |

### 결과 해석

**1. 틀린 참조 답변 2개가 정확히 잡혔습니다**
- 3번(RNN 비교)과 5번(멀티헤드 어텐션)의 참조 답변을 일부러 틀리게 작성했는데, `correctness`와 `accuracy` 모두 이를 감지했습니다.
- RAG 시스템은 컨텍스트에 근거하여 올바르게 답변했지만, **참조 답변이 틀리므로** 불일치로 판정된 것입니다.

**2. completeness는 다른 관점을 봅니다**
- 멀티헤드 어텐션 질문은 `accuracy=N`이지만 `completeness=Y`입니다. 참조 답변의 핵심 키워드를 생성 답변이 충분히 다루고 있기 때문이에요.
- 반면 포지셔널 인코딩은 `accuracy=Y`인데 `completeness=N` — 사실은 맞지만 참조 답변보다 내용이 부족하다고 판단한 것입니다.

**3. relevance는 모든 질문에서 Y**
- 답변 내용이 맞든 틀리든, **질문 주제에서 벗어나지 않으면** Y입니다. 참조 답변과의 일치 여부와 무관합니다.

**4. 임베딩 거리로 보는 의미 차이**
- 틀린 참조 답변(RNN 비교: 0.19)이 올바른 참조 답변 일부(아키텍처: 0.48)보다 거리가 **더 가까운** 경우도 있습니다. 이는 틀린 답변이라도 관련 단어를 많이 공유하면 임베딩 거리가 작아질 수 있다는 점을 보여줍니다.
- → 임베딩 거리만으로 정확성을 판단하면 안 되며, LLM-as-Judge와 함께 사용해야 합니다.

> 💡 **실무 팁**: 평가 데이터셋의 참조 답변 품질이 곧 평가의 신뢰도입니다. 틀린 참조 답변이 섞여 있으면 올바른 시스템 답변이 낮은 점수를 받게 돼요. 데이터셋을 만들 때 **참조 답변의 정확성을 반드시 검증**하세요.

---

## 정리

### LangSmith 대시보드 항목 설명

LangSmith 대시보드의 실험(Experiment) 목록에서 확인할 수 있는 항목들입니다:

| 항목 | 설명 |
|------|------|
| **P50 Latency** | 전체 Run의 중간값(50번째 백분위) 응답 시간 — 일반적인 응답 속도 |
| **P99 Latency** | 99번째 백분위 응답 시간 — 가장 느린 요청의 근사치 |
| **Run Count** | 평가 대상으로 실행된 총 Run 수 (= 데이터셋 예제 수) |
| **Error Rate** | 실행 중 오류가 발생한 비율 |
| **{...} metadata** | `evaluate()` 호출 시 `metadata=`로 전달한 실험 조건 (모델명, 청크 크기 등) |

> P50과 P99의 차이가 크다면 일부 요청이 비정상적으로 느린 것이므로, 해당 Run을 클릭해서 어느 단계에서 병목이 발생했는지 확인하세요.

### LangSmith 평가 핵심 흐름

1. `Client()`로 LangSmith에 연결
2. `create_dataset()` + `create_example()`로 데이터셋 업로드
3. 평가할 시스템을 `inputs -> outputs` 함수로 래핑
4. `LangChainStringEvaluator`로 평가자 정의
5. `evaluate()`로 실험 실행 → LangSmith 대시보드에서 확인

### 이번 노트북에서 다룬 평가자

| 평가자 | 유형 | 특징 |
|--------|------|------|
| `qa` | LLM-as-Judge | 가장 기본적인 정확성 평가 (CORRECT/INCORRECT) |
| `labeled_criteria` | LLM-as-Judge | 사용자 정의 기준으로 평가 (기준별 1개 인스턴스) |
| `embedding_distance` | 임베딩 유사도 | LLM 호출 없이 벡터 거리 비교 (비용 절감) |

### 참고: 추가 평가자

| 평가자 | 설명 |
|--------|------|
| `cot_qa` | Chain-of-Thought 추론 후 정확성 판정 (qa보다 정밀) |
| `context_qa` | 검색된 Context를 reference로 사용하여 평가 |
| `criteria` | 참조 답변 없이 단일 기준(conciseness, harmfulness 등)으로 평가 |
| `labeled_score_string` | 참조 답변 대비 1-10 점수 반환 (정규화 가능) |

### RAGAS vs LangSmith

| 특징 | RAGAS | LangSmith |
|------|-------|-----------|
| 초점 | RAG 특화 4가지 지표 | 범용 평가 + 프로덕션 모니터링 |
| 평가 방식 | RAG 메트릭 | LLM-as-Judge, 커스텀 |
| 데이터셋 | 로컬 관리 | 클라우드 관리 |
| 시각화 | Python 코드 | 웹 대시보드 |
| 추적 | 없음 | 실시간 트레이싱 |

### LLM-as-Judge 장단점

**장점**: 자동화, 확장 가능, 일관된 기준 적용<br/>
**단점**: 평가 LLM 편향 가능성, 추가 API 비용, 복잡한 케이스 오판 가능

### 다음 노트북 예고

LangSmith가 제공하는 기본 평가자 외에 **직접 커스텀 평가 로직**을 작성하는 방법을 배워볼게요. 규칙 기반, LLM 기반, 임베딩 기반 세 가지 유형을 직접 구현합니다.